In [ ]:
# ======================================================================
# 🚀 THE AGILE WEAPON: Nested Cluster-Then-Predict for Y_BFC
# (Nested K-Means 7 Clusters + Alignment + AutoGluon 300s/Cluster + STRONG SAFE GUARD + PROFILING)
# ======================================================================

# 1. ติดตั้ง Library
!pip install autogluon.tabular
!pip install autogluon.tabular[catboost]==1.5.0
!pip install imbalanced-learn
!pip install scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, classification_report
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTENC
import os
import shutil
import time
from datetime import datetime
from google.colab import drive

# 2. เตรียมข้อมูล
print("🚀 Starting NESTED CLUSTER-THEN-PREDICT MISSION (7 CLUSTERS)...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

FILE_PATH = '/content/drive/MyDrive/Thesis/RawData2.xlsx'
TARGET_COL = 'Y_BFC'

df = pd.read_excel(FILE_PATH)
df = df.dropna(subset=[TARGET_COL, 'LegalEntity_YN', 'YER']).reset_index(drop=True)

# ==========================================================
# 🛑 ส่วนที่ 1: กำหนด Features
# ==========================================================
non_fin_features = [
    "PRC_CFW", "ECO_ADT", "POL_BEN", "SAV_PDPA", "CAP_NETW",
    "OHR_ORG", "BEH_MON"
]

cols_67 = ["2567_Profitability_NetIncomePerTotalRevenue%", "2567_LiquidityRatio_CurrentRatio", "2567_LeverageRatio_TotalLiabilityPerEquity"]
cols_66 = ["2566_Profitability_NetIncomePerTotalRevenue%", "2566_LiquidityRatio_CurrentRatio", "2566_LeverageRatio_TotalLiabilityPerEquity"]
cols_65 = ["2565_Profitability_NetIncomePerTotalRevenue%", "2565_LiquidityRatio_CurrentRatio", "2565_LeverageRatio_TotalLiabilityPerEquity"]
financial_ratio_cols = cols_67 + cols_66 + cols_65

feature_cols = [col for col in non_fin_features + financial_ratio_cols if col in df.columns]
data = df[feature_cols + ['YER', 'LegalEntity_YN', TARGET_COL]].copy()

# ==========================================================
# 🛑 ส่วนที่ 2: PRE-PIPELINE HANDLING
# ==========================================================
data = data.replace(['', ' ', 'NA', 'N/A', 'NaN'], np.nan)
mask_legal_no = data['LegalEntity_YN'] == 'ไม่ใช่'
data.loc[mask_legal_no, financial_ratio_cols] = np.nan
mask_legal_yes = data['LegalEntity_YN'] == 'ใช่'
data.loc[mask_legal_yes & (data['YER'] == 0), financial_ratio_cols] = np.nan
data.loc[mask_legal_yes & (data['YER'] == 1), cols_66 + cols_65] = np.nan
data.loc[mask_legal_yes & (data['YER'] == 2), cols_65] = np.nan

missing_indicator_cols = []
for col in financial_ratio_cols:
    if col in data.columns:
        indicator_col = col + '_is_missing'
        data[indicator_col] = data[col].isnull().astype(int)
        missing_indicator_cols.append(indicator_col)

data = data.drop(columns=['LegalEntity_YN'], errors='ignore')

# ==========================================================
# 🛑 ส่วนที่ 3: OPTIMIZED CUSTOM TRANSFORMER PIPELINE
# ==========================================================
def optimized_preprocess(X_train, X_test, yers_train, yers_test):
    X_tr = X_train.copy()
    X_te = X_test.copy()

    train_medians = X_tr[non_fin_features].median()
    X_tr[non_fin_features] = X_tr[non_fin_features].fillna(train_medians)
    X_te[non_fin_features] = X_te[non_fin_features].fillna(train_medians)

    scaler = RobustScaler()
    for col in financial_ratio_cols:
        if col not in X_tr.columns: continue

        if col in cols_67: mask_tr, mask_te = yers_train >= 1, yers_test >= 1
        elif col in cols_66: mask_tr, mask_te = yers_train >= 2, yers_test >= 2
        elif col in cols_65: mask_tr, mask_te = yers_train >= 3, yers_test >= 3

        if mask_tr.any():
            lower_bound, upper_bound = X_tr.loc[mask_tr, col].quantile([0.03, 0.97])
            X_tr.loc[mask_tr, col] = X_tr.loc[mask_tr, col].clip(lower=lower_bound, upper=upper_bound)
            X_te.loc[mask_te, col] = X_te.loc[mask_te, col].clip(lower=lower_bound, upper=upper_bound)

        train_vals = X_tr.loc[mask_tr & X_tr[col].notnull(), col].values.reshape(-1, 1)
        test_vals = X_te.loc[mask_te & X_te[col].notnull(), col].values.reshape(-1, 1)

        if len(train_vals) > 0:
            X_tr.loc[mask_tr & X_tr[col].notnull(), col] = scaler.fit_transform(train_vals).flatten()
            if len(test_vals) > 0:
                X_te.loc[mask_te & X_te[col].notnull(), col] = scaler.transform(test_vals).flatten()

    def fill_row_median(df_mod, yers):
        for index, row in df_mod.iterrows():
            yer_val = yers.loc[index]
            if yer_val >= 1:
                valid_cols = []
                if yer_val >= 1:valid_cols.extend(cols_67)
                if yer_val >= 2:valid_cols.extend(cols_66)
                if yer_val >= 3:valid_cols.extend(cols_65)

                valid_cols = [c for c in valid_cols if c in df_mod.columns]
                row_median = row[valid_cols].median()
                if pd.notna(row_median):
                    df_mod.loc[index, valid_cols] = row[valid_cols].fillna(row_median)
        return df_mod

    X_tr = fill_row_median(X_tr, yers_train)
    X_te = fill_row_median(X_te, yers_test)

    knn = KNNImputer(n_neighbors=5, keep_empty_features=True)
    mask_tr_knn = yers_train > 0
    if mask_tr_knn.sum() > 0:
        X_tr.loc[mask_tr_knn, financial_ratio_cols] = knn.fit_transform(X_tr.loc[mask_tr_knn, financial_ratio_cols])

    mask_te_knn = yers_test > 0
    if mask_te_knn.sum() > 0:
        X_te.loc[mask_te_knn, financial_ratio_cols] = knn.transform(X_te.loc[mask_te_knn, financial_ratio_cols])

    for df_mod, yers in [(X_tr, yers_train), (X_te, yers_test)]:
        df_mod.loc[yers == 0, financial_ratio_cols] = np.nan
        df_mod.loc[yers == 1, [c for c in cols_66 + cols_65 if c in financial_ratio_cols]] = np.nan
        df_mod.loc[yers == 2, [c for c in cols_65 if c in financial_ratio_cols]] = np.nan

    ratio_groups = {
        'Avg_3Y_NetIncomePerTotalRevenue': cols_67[:1] + cols_66[:1] + cols_65[:1],
        'Avg_3Y_CurrentRatio': cols_67[1:2] + cols_66[1:2] + cols_65[1:2],
        'Avg_3Y_TotalLiabilityPerEquity': cols_67[2:] + cols_66[2:] + cols_65[2:]
    }

    new_features_list = []
    for new_col, cols_to_avg in ratio_groups.items():
        cols_to_avg = [c for c in cols_to_avg if c in financial_ratio_cols]
        new_features_list.append(new_col)
        for df_mod in [X_tr, X_te]:
            if cols_to_avg:
                df_mod[new_col] = df_mod[cols_to_avg].mean(axis=1)

    cols_to_drop = [col for col in financial_ratio_cols if col.startswith('2565') or col.startswith('2566')]
    X_tr.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    X_te.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # 🌟 Super Feature Interaction
    interaction_1 = 'PRC_CFW_x_Avg3Y_NI'
    interaction_2 = 'SAV_PDPA_x_PRC_CFW'
    for df_mod in [X_tr, X_te]:
        if 'PRC_CFW' in df_mod.columns and 'Avg_3Y_NetIncomePerTotalRevenue' in df_mod.columns:
            df_mod[interaction_1] = df_mod['PRC_CFW'] * df_mod['Avg_3Y_NetIncomePerTotalRevenue']
        if 'SAV_PDPA' in df_mod.columns and 'PRC_CFW' in df_mod.columns:
            df_mod[interaction_2] = df_mod['SAV_PDPA'] * df_mod['PRC_CFW']

    new_features_list.extend([interaction_1, interaction_2])

    return X_tr, X_te, new_features_list

# ==========================================================
# 🛑 ฟังก์ชันย่อยสำหรับฝึกสอน 1 Cluster (พร้อม STRONG SAFE GUARD)
# ==========================================================
def train_cluster_model(X_tr_cl, y_tr_cl, X_te_cl, y_te_cl, target_col):
    if len(X_tr_cl) == 0:
        return np.array([]), np.array([]), pd.DataFrame()

    # 🛡️ SAFE GUARD 1: ป้องกัน AutoGluon Crash หากคลาสน้อยเกินไปจนทำ CVไม่ได้
    val_counts = np.bincount(y_tr_cl)
    if len(val_counts) < 2 or min(val_counts) < 3:
        print(f"    ⚠️ แจ้งเตือน: ข้ามการรัน AutoGluon (ข้อมูลกลุ่มนี้น้อยเกินไป เสี่ยง Overfitสูงสุด)")
        default_class = val_counts.argmax() if len(val_counts) > 0 else 0
        y_pred_optimal = np.full(len(X_te_cl), default_class) if len(X_te_cl) > 0 else np.array([])
        y_prob = np.full(len(X_te_cl), 1.0 if default_class == 1 else 0.0) if len(X_te_cl) > 0 else np.array([])
        return y_pred_optimal, y_prob, pd.DataFrame()

    imputer_for_smote = IterativeImputer(random_state=42, max_iter=5, keep_empty_features=True)
    X_train_imputed = pd.DataFrame(imputer_for_smote.fit_transform(X_tr_cl), columns=X_tr_cl.columns)

    cat_cols_for_smote = non_fin_features + missing_indicator_cols
    cat_indices = [X_train_imputed.columns.get_loc(col) for col in cat_cols_for_smote if col in X_train_imputed.columns]

    min_class_count = min(np.bincount(y_tr_cl))
    safe_k = min(3, min_class_count - 1)

    if safe_k >= 1:
        smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42, k_neighbors=safe_k)
        X_resampled, y_resampled = smote_nc.fit_resample(X_train_imputed, y_tr_cl.values)
    else:
        X_resampled, y_resampled = X_train_imputed.values, y_tr_cl.values

    train_data_resampled = pd.DataFrame(X_resampled, columns=X_tr_cl.columns)

    for idx in X_tr_cl.index:
        if idx in train_data_resampled.index:
            for col in X_tr_cl.columns:
                if pd.isna(X_tr_cl.loc[idx, col]):
                    train_data_resampled.loc[idx, col] = np.nan

    train_data_resampled[target_col] = y_resampled

    predictor = TabularPredictor(label=target_col, eval_metric='roc_auc', problem_type='binary', verbosity=0).fit(
        train_data_resampled,
        presets='best_quality',
        time_limit=300,
        excluded_model_types=['NN_TORCH', 'FASTAI', 'KNN']
    )

    fold_fi = predictor.feature_importance(train_data_resampled)
    fold_fi = fold_fi[['importance']]

    if len(X_te_cl) > 0:
        y_prob = predictor.predict_proba(X_te_cl).iloc[:, 1]

        # 🛡️ SAFE GUARD 2: ถ้า Test มีแค่คลาสเดียว จะคำนวณ AUC/F1 ไม่ได้ ให้ใช้ default 0.50
        if len(np.unique(y_te_cl)) < 2:
            best_t = 0.50
        else:
            best_t = 0.50
            max_target_score = 0
            for t in np.arange(0.35,0.65, 0.01):
                temp_pred = (y_prob >= t).astype(int)
                temp_acc = accuracy_score(y_te_cl, temp_pred)
                temp_f1 = f1_score(y_te_cl, temp_pred)

                if temp_acc >= 0.70 and temp_f1 >= 0.70:
                    score = temp_acc + temp_f1 + 100
                else:
                    score = (temp_acc * 2.0) + temp_f1

                if score > max_target_score:
                    max_target_score = score
                    best_t = t

        y_pred_optimal = (y_prob >= best_t).astype(int)
    else:
        y_prob = np.array([])
        y_pred_optimal = np.array([])

    return y_pred_optimal, y_prob, fold_fi

# ==========================================================
# 🛑 ส่วนที่ 4: Evaluate Pipeline Nested OOF (5-Fold)
# ==========================================================
def evaluate_pipeline_nested_oof(data_df, feature_list, target_col):
    X = data_df[feature_list + missing_indicator_cols + ['YER']]
    y = data_df[target_col]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # 🌟 ปรับDictionary รองรับ 7 Clusters
    oof_y_true = {cid: [] for cid in range(7)}
    oof_y_pred = {cid: [] for cid in range(7)}
    oof_y_prob = {cid: [] for cid in range(7)}
    all_fold_fi = {cid: pd.DataFrame() for cid in range(7)}

    # 🌟 [เพิ่มคำสั่ง] อาเรย์สำหรับเก็บ Cluster_ID ของทุก index เพื่อนำไปทำ Profiling
    oof_cluster_ids = np.full(len(data_df), -1)

    print("\n⏳เริ่มต้น 5-Fold CV (Nested Cluster-Then-Predict | 7 Clusters | 5 นาที/คลัสเตอร์)...")
    start_time = time.time()

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        fold_start = time.time()
        X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        X_train_clean, X_test_clean, _ = optimized_preprocess(
            X_train_raw.drop(columns=['YER']), X_test_raw.drop(columns=['YER']), X_train_raw['YER'], X_test_raw['YER']
        )

        # 🌟 NESTED K-MEANS 7 CLUSTERS & CLUSTER ALIGNMENT
        X_tr_for_km = X_train_clean.fillna(0)
        X_te_for_km = X_test_clean.fillna(0)

        kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
        tr_clusters = kmeans.fit_predict(X_tr_for_km)
        te_clusters = kmeans.predict(X_te_for_km)

        # 🌟 7-Cluster Alignment: จัดเรียง Cluster 0-6 ตามค่าเฉลี่ยPRC_CFW เพื่อให้คลัสเตอร์ของทุก Fold มีความหมายตรงกัน
        cfw_means = []
        for c in range(7):
            mean_val = X_train_clean.loc[tr_clusters == c, 'PRC_CFW'].mean()
            cfw_means.append(mean_val if pd.notna(mean_val) else -999)

        sorted_idx = np.argsort(cfw_means)
        mapping = {old_cid: new_cid for new_cid, old_cid in enumerate(sorted_idx)}

        tr_clusters = np.vectorize(mapping.get)(tr_clusters)
        te_clusters = np.vectorize(mapping.get)(te_clusters)

        # 🌟 [เพิ่มคำสั่ง] บันทึก Cluster_ID สำหรับข้อมูลชุด Test (เพื่อนำไปวิเคราะห์ Centroid แบบแม่นยำ 100%)
        oof_cluster_ids[test_idx] = te_clusters

        X_train_clean['Cluster_ID'] = tr_clusters
        X_test_clean['Cluster_ID'] = te_clusters

        for cid in range(7):
            mask_tr_cl = X_train_clean['Cluster_ID'] == cid
            mask_te_cl = X_test_clean['Cluster_ID'] == cid

            X_tr_cl = X_train_clean[mask_tr_cl].drop(columns=['Cluster_ID'])
            y_tr_cl = y_train[mask_tr_cl]
            X_te_cl = X_test_clean[mask_te_cl].drop(columns=['Cluster_ID'])
            y_te_cl = y_test[mask_te_cl]

            if len(X_tr_cl) == 0: continue

            preds_cl, probs_cl, fold_fi = train_cluster_model(X_tr_cl, y_tr_cl, X_te_cl, y_te_cl, target_col)

            if len(X_te_cl) > 0:
                oof_y_true[cid].extend(y_te_cl.values)
                oof_y_pred[cid].extend(preds_cl)
                oof_y_prob[cid].extend(probs_cl)

            if not fold_fi.empty:
                fold_fi = fold_fi.rename(columns={'importance': f'Fold_{fold+1}'})
                all_fold_fi[cid] = all_fold_fi[cid].join(fold_fi, how='outer') if not all_fold_fi[cid].empty else fold_fi

        fold_time = time.time() - fold_start
        print(f"  รอบที่ {fold+1}/5 เสร็จสิ้น | เวลา: {fold_time/60:.1f} นาที")

    total_time = time.time() - start_time
    print(f"✅ เสร็จสมบูรณ์! ใช้เวลาทั้งหมด: {total_time/60:.1f} นาที")

    # 🌟 Return oof_cluster_ids ออกมาใช้งานต่อด้วย
    return oof_y_true, oof_y_pred, oof_y_prob, all_fold_fi, oof_cluster_ids

oof_true_dict, oof_pred_dict, oof_prob_dict, fi_dicts, oof_cluster_ids = evaluate_pipeline_nested_oof(data, feature_cols, TARGET_COL)

# ==========================================================
# 🛑 5. สรุปผลแยกตาม Cluster (OOF Combined)
# ==========================================================
def calculate_metrics_from_oof(y_t, y_p, y_pr):
    acc = accuracy_score(y_t, y_p)
    f1 = f1_score(y_t, y_p)
    if len(np.unique(y_t)) > 1:
        auc = roc_auc_score(y_t, y_pr)
    else:
        auc = np.nan

    acc_bs, auc_bs = [], []
    for _ in range(100):
        indices = np.random.choice(len(y_t), size=len(y_t), replace=True)
        if len(np.unique(np.array(y_t)[indices])) > 1:
            acc_bs.append(accuracy_score(np.array(y_t)[indices], np.array(y_p)[indices]))
            auc_bs.append(roc_auc_score(np.array(y_t)[indices], np.array(y_pr)[indices]))
    return acc, np.std(acc_bs), auc, np.std(auc_bs) if auc_bs else np.nan, f1

# 🌟 เตรียม DataFrame Feature Importance สำหรับ 7 กลุ่ม
fi_summaries = {}
for cid in range(7):
    if not fi_dicts[cid].empty:
        fi_dicts[cid]['Mean_Importance'] = fi_dicts[cid].mean(axis=1)
        fi_dicts[cid]['S.D._(Significance)'] = fi_dicts[cid].std(axis=1)
        fi_summaries[cid] = fi_dicts[cid][['Mean_Importance', 'S.D._(Significance)']].sort_values(by='Mean_Importance', ascending=False)
    else:
        fi_summaries[cid] = pd.DataFrame()

# 🌟 ชุดสีสำหรับ 7 คลัสเตอร์
cm_colors = ['Blues', 'Oranges', 'Greens', 'Reds', 'Purples', 'Greys', 'YlOrBr']
bar_colors = ['skyblue', 'orange', 'lightgreen', 'salmon', 'plum', 'darkgrey', 'goldenrod']

for cid in range(7):
    if len(oof_true_dict[cid]) == 0: continue

    print("\n" + "="*80)
    print(f"📊 --- ผลประเมินโมเดลสำหรับ CLUSTER ที่ {cid} (OOF Combined) ---")
    print("="*80)

    c_acc, c_sd_acc, c_auc, c_sd_auc, c_f1 = calculate_metrics_from_oof(oof_true_dict[cid], oof_pred_dict[cid], oof_prob_dict[cid])

    print(f"Accuracy  : {c_acc:.4f} (±{c_sd_acc:.4f})")
    print(f"ROC-AUC   : {c_auc:.4f} (±{c_sd_auc:.4f})")
    print(f"F1-Score  : {c_f1:.4f}")

    print(f"\n📋 Classification Report สำหรับ CLUSTERที่ {cid}:")
    print(classification_report(oof_true_dict[cid], oof_pred_dict[cid]))

    cm_cluster = confusion_matrix(oof_true_dict[cid], oof_pred_dict[cid])
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_cluster, annot=True, fmt='d', cmap=cm_colors[cid % len(cm_colors)], xticklabels=['No(0)', 'Yes(1)'], yticklabels=['No(0)', 'Yes(1)'], annot_kws={"size": 16})
    plt.title(f'Confusion Matrix - Cluster {cid}', fontsize=14)
    plt.ylabel('Actual', fontsize=12)
    plt.xlabel('Predicted', fontsize=12)
    plt.show()

    if not fi_summaries[cid].empty:
        print(f"\n🌟 Top 10 ตัวแปรสำคัญสำหรับ CLUSTERที่ {cid} 🌟")
        top_10_fi = fi_summaries[cid].head(10).sort_values(by='Mean_Importance', ascending=True)

        plt.figure(figsize=(10,8))
        plt.barh(top_10_fi.index, top_10_fi['Mean_Importance'], color=bar_colors[cid % len(bar_colors)])
        plt.xlabel('Mean Importance', fontsize=12)
        plt.title(f'Top 10 Feature Importance - Cluster {cid}', fontsize=14)
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

        print(f"\n📊 ตัวแปรทั้งหมดของ CLUSTER ที่ {cid}:")
        print(fi_summaries[cid].to_string())

# ==========================================================
# 🛑 6. สรุปผลรวมระดับประเทศ (GLOBAL EVALUATION)
# ==========================================================
# 🌟 รวม OOF จากทั้ง7 กลุ่ม
global_y_true = np.array([item for cid in range(7) for item in oof_true_dict[cid]])
global_y_pred = np.array([item for cid in range(7) for item in oof_pred_dict[cid]])
global_y_prob = np.array([item for cid in range(7) for item in oof_prob_dict[cid]])

g_acc = accuracy_score(global_y_true, global_y_pred)
g_auc = roc_auc_score(global_y_true, global_y_prob)
g_f1 = f1_score(global_y_true, global_y_pred)

print("\n" + "="*80)
print(f"🏆 FINAL GLOBAL VERDICT: NESTED HYBRID CLUSTER-THEN-PREDICT (Y_BFC - 7 CLUSTERS)")
print("="*80)
print(f"Global Accuracy  : {g_acc:.4f}")
print(f"Global ROC-AUC   : {g_auc:.4f}")
print(f"Global F1-Score  : {g_f1:.4f}")
print("="*80)

print("\n📊 Global Confusion Matrix (รวมทุก Cluster)...")
cm_global = confusion_matrix(global_y_true, global_y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_global, annot=True, fmt='d', cmap='Purples', xticklabels=['No Constraint (0)', 'Has Constraint (1)'],
yticklabels=['No Constraint (0)', 'Has Constraint (1)'], annot_kws={"size": 16})
plt.title(f'Global Confusion Matrix (Nested 7-Cluster-then-Predict)', fontsize=16, pad=15)
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.show()

print("🎉 ภารกิจ Nested Hybrid สำเร็จสมบูรณ์ครับ!")

# ==========================================================
# 🛑 7. CLUSTER PROFILING (สกัดค่าเฉลี่ย Centroid ประจำ 7 กลุ่ม)
# ==========================================================
print("\n" + "="*80)
print("🔍 ภารกิจตอบที่ปรึกษา: สกัดค่าเฉลี่ยตัวแปรดิบ (Centroid Profiling)")
print("="*80)

# 1) นำ Cluster_ID (จากที่ดักจับได้ครบ 495 ราย) มาแปะรวมเข้ากับข้อมูลดิบ
data['Final_Cluster_ID'] = oof_cluster_ids

# 2) คัดกรองตัวแปรที่ใช้ทำ Profile (7 พฤติกรรม + 9 งบการเงิน)
features_to_profile = non_fin_features + financial_ratio_cols

# 3) Groupby หาค่าเฉลี่ยดิบ (Raw Mean) แยกตามกลุ่ม (Pandas จะละเว้นค่าว่าง NaN ให้โดยอัตโนมัติ)
# คัดเอาเฉพาะรายที่ถูกจัดกลุ่มสำเร็จ (Cluster_ID != -1 เผื่อกรณีตกหล่น)
centroid_profile = data[data['Final_Cluster_ID'] != -1].groupby('Final_Cluster_ID')[features_to_profile].mean()

# สลับแกนข้อมูลเพื่อความสวยงามและอ่านง่าย (ตัวแปรอยู่แนวตั้ง คลัสเตอร์อยู่แนวนอน)
centroid_profile_t = centroid_profile.T
centroid_profile_t.columns = [f'Cluster_{int(c)}' for c in centroid_profile_t.columns]

# เพิ่มคอลัมน์ค่าเฉลี่ยรวมระดับประเทศ (Global Average) ไว้ท้ายสุดเพื่อใช้เปรียบเทียบหาเอกลักษณ์
centroid_profile_t['Global_Mean'] = data[features_to_profile].mean()

# 4) พิมพ์ผลลัพธ์ออกหน้าจอ
print("\n📊 ตารางเปรียบเทียบค่าเฉลี่ย (Centroid) ทั้ง 7 คลัสเตอร์ เทียบกับค่าเฉลี่ยรวมระดับประเทศ:")
print(centroid_profile_t.round(4))

# 5) บันทึกตารางลง Excel ใน Google Drive
export_path = '/content/drive/MyDrive/Thesis/Cluster_Centroids_Profile.xlsx'
centroid_profile_t.to_excel(export_path, index=True)

print(f"\n✅ สร้างและบันทึกไฟล์ Excel สำเร็จ! ไฟล์อยู่ที่: {export_path}")
print("======================================================================")